``eval_features`` scores a feature set by cross-validated classification performance. The default reproduces the linear-SVM, leave-one-out, balanced-accuracy benchmark used throughout the analysis: ``balanced_accuracy_score(y, cross_val_predict(SVC(kernel='linear'), X, y, cv=LeaveOneOut())) * 100``.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
import aaanalysis as aa

aa.options["verbose"] = False
# A small two-class feature matrix (rows = proteins, columns = features)
X, labels = make_classification(n_samples=40, n_features=8, n_informative=5,
                                n_redundant=1, random_state=0)

# Default: linear SVM + leave-one-out + balanced accuracy (percentage)
score = aa.eval_features(X, labels)
print(f"balanced accuracy = {score:.1f}%")

balanced accuracy = 64.7%


Swap in any scikit-learn estimator via ``model``, a CV splitter or integer via ``cv``, and a metric name via ``metric``. ``random_state`` makes stochastic estimators (e.g. random forest) reproducible.

In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

rows = {
    "SVM / LOO / bACC": aa.eval_features(X, labels),
    "SVM / 5-fold / accuracy": aa.eval_features(X, labels, cv=5, metric="accuracy"),
    "RF / 5-fold / bACC": aa.eval_features(X, labels, model=RandomForestClassifier(n_estimators=100),
                                           cv=StratifiedKFold(5, shuffle=True, random_state=42),
                                           random_state=42),
}
df_eval = pd.DataFrame({"score [%]": rows}).round(1)
aa.display_df(df_eval)

,score [%]
SVM / LOO / bACC,64.700000
SVM / 5-fold / accuracy,60.000000
RF / 5-fold / bACC,80.200000


``mask_known_pos`` selects the Positive-Unlabeled (PU) mask-known-positives CV variant: masked known positives still train every fold but are never scored as test points, so the score reflects only the held-out, non-trivial samples.

In [3]:
mask = np.zeros(len(labels), dtype=bool)
mask[np.where(labels == 1)[0][:5]] = True   # treat 5 positives as "known"
masked = aa.eval_features(X, labels, mask_known_pos=mask)
print(f"PU mask-known-positives balanced accuracy = {masked:.1f}%")

PU mask-known-positives balanced accuracy = 64.3%
